In [2]:
import json
import pandas as pd
import glob
import os

def parse_ava_logs(directory_path):
    all_data = []
    
    # Locate all JSON files in the specified directory [cite: 9, 10]
    file_pattern = os.path.join(directory_path, "*.json")
    files = glob.glob(file_pattern)
    
    for file_path in files:
        with open(file_path, 'r') as f:
            try:
                data = json.load(f)
                # Handle cases where the file might be a single object or a list [cite: 9]
                interactions = data if isinstance(data, list) else [data]
            except json.JSONDecodeError:
                continue

            for inter in interactions:
                # Initialize record with core metadata 
                record = {
                    "interaction_id": inter.get("id"),
                    "session_id": inter.get("sessionId"),
                    "timestamp": inter.get("timestamp"),
                    "use_case": os.path.basename(file_path).replace('.json', ''),
                    "guardrail_ms": 0,
                    "routing_ms": 0,
                    "orchestration_ms": 0,
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "reasoning_steps": 0
                }
                
                # Parse the 'output' array for traces and reasoning 
                for step in inter.get('output', []):
                    step_type = step.get('type')
                    trace = step.get('trace', {})
                    
                    # Extract Latency and Tokens from metadata
                    metadata = None
                    if 'metadata' in trace:
                        metadata = trace['metadata']
                    elif 'modelInvocationOutput' in trace:
                        metadata = trace['modelInvocationOutput'].get('metadata')
                    
                    if metadata:
                        ms = metadata.get('totalTimeMs', 0)
                        if step_type == 'guardrail_trace':
                            record["guardrail_ms"] += ms
                        elif step_type == 'routing_classifier_trace':
                            record["routing_ms"] += ms
                        elif step_type == 'orchestration_trace':
                            record["orchestration_ms"] += ms
                        
                        usage = metadata.get('usage', {})
                        record["input_tokens"] += usage.get('inputTokens', 0)
                        record["output_tokens"] += usage.get('outputTokens', 0)

                    # Count reasoning/rationale steps 
                    if 'rationale' in trace or 'rationale' in trace.get('modelInvocationOutput', {}):
                        record["reasoning_steps"] += 1
                
                # Calculate total interaction time
                record["total_latency_ms"] = record["guardrail_ms"] + record["routing_ms"] + record["orchestration_ms"]
                all_data.append(record)
                
    return pd.DataFrame(all_data)


df = parse_ava_logs('./test_dataset')
print(df.head())

                     interaction_id                            session_id  \
0  324c3c4832f590e0db98efaa78fcc815  5bef0490-08c0-11f1-907e-cdcae5aa3470   
1  fb328a876ccc595477ac5893b8325ac6  5bef0490-08c0-11f1-907e-cdcae5aa3470   
2  e1cfc316145cc7fb0efe1cecbf5e4c9d  5bef0490-08c0-11f1-907e-cdcae5aa3470   
3  f18779ccb5fd17a66be0609de7635f13  5bef0490-08c0-11f1-907e-cdcae5aa3470   
4  5eb768f42fa89941d31e9ddfae4740ed  5bef0490-08c0-11f1-907e-cdcae5aa3470   

                  timestamp     use_case  guardrail_ms  routing_ms  \
0  2026-02-13T09:51:54.991Z  case_create             0           0   
1  2026-02-13T09:51:23.907Z  case_create          1300        1448   
2  2026-02-13T09:49:40.287Z  case_create          2842        1757   
3  2026-02-13T09:46:11.102Z  case_create          2498        1136   
4  2026-02-13T09:45:13.224Z  case_create          2895        1681   

   orchestration_ms  input_tokens  output_tokens  reasoning_steps  \
0             17134         22585            70

In [3]:
df

,interaction_id,session_id,timestamp,use_case,guardrail_ms,routing_ms,orchestration_ms,input_tokens,output_tokens,reasoning_steps,total_latency_ms
0,324c3c4832f590e0db98efaa78fcc815,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:51:54.991Z,case_create,0,0,17134,22585,701,2,17134
1,fb328a876ccc595477ac5893b8325ac6,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:51:23.907Z,case_create,1300,1448,10807,26468,325,2,13555
2,e1cfc316145cc7fb0efe1cecbf5e4c9d,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:49:40.287Z,case_create,2842,1757,30502,43339,1348,4,35101
3,f18779ccb5fd17a66be0609de7635f13,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:46:11.102Z,case_create,2498,1136,28446,39011,1052,4,32080
4,5eb768f42fa89941d31e9ddfae4740ed,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:45:13.224Z,case_create,2895,1681,24342,26580,835,3,28918
5,8c498d0fe55194455e35c136f84bc222,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:44:16.655Z,case_create,3030,1376,15478,24562,578,3,19884
6,30cb8862512870f21004590edf80082b,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:43:15.547Z,case_create,3192,1516,15600,23048,503,3,20308
7,da8c8e9e4c62c2bdab42928f052defdf,5bef0490-08c0-11f1-907e-cdcae5aa3470,2026-02-13T09:42:50.197Z,case_create,2741,1878,5929,8594,228,1,10548
8,7964a6fdff7bfb0333329e5516211ec9,ef2fc940-08c2-11f1-907e-cdcae5aa3470,2026-02-13T10:02:02.713Z,case_update,3010,1904,20781,40269,770,5,25695
9,8da7d83ff125d2e800d353f3b58a6158,ef2fc940-08c2-11f1-907e-cdcae5aa3470,2026-02-13T10:01:16.170Z,case_update,2702,2153,19073,29780,622,4,23928


In [4]:
import json
import pandas as pd

# Load JSON file
with open("test_dataset/case_create.json", "r") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

df

,id,projectId,timestamp,tags,bookmarked,name,release,version,userId,environment,...,totalCost,level,errorCount,warningCount,defaultCount,debugCount,observationCount,inputTokens,outputTokens,totalTokens
0,324c3c4832f590e0db98efaa78fcc815,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:51:54.991Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.078269999999,DEFAULT,0,0,1,0,1,22585,701,23286
1,fb328a876ccc595477ac5893b8325ac6,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:51:23.907Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.084279,DEFAULT,0,0,1,0,1,26468,325,26793
2,e1cfc316145cc7fb0efe1cecbf5e4c9d,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:49:40.287Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.150236999999,DEFAULT,0,0,1,0,1,43339,1348,44687
3,f18779ccb5fd17a66be0609de7635f13,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:46:11.102Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.132813,DEFAULT,0,0,1,0,1,39011,1052,40063
4,5eb768f42fa89941d31e9ddfae4740ed,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:45:13.224Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.092265,DEFAULT,0,0,1,0,1,26580,835,27415
5,8c498d0fe55194455e35c136f84bc222,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:44:16.655Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.082356,DEFAULT,0,0,1,0,1,24562,578,25140
6,30cb8862512870f21004590edf80082b,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:43:15.547Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.076689,DEFAULT,0,0,1,0,1,23048,503,23551
7,da8c8e9e4c62c2bdab42928f052defdf,cmgt9niok000wvo07s9ywpqc3,2026-02-13T09:42:50.197Z,[],False,Invoke Agent,9.0 Hotfix,None,username@email.com,staging,...,0.029202,DEFAULT,0,0,1,0,1,8594,228,8822


In [5]:
# Load JSON file
with open("test_dataset/license_management.json", "r") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data)

df

,id,projectId,timestamp,tags,bookmarked,name,release,version,userId,environment,...,totalCost,level,errorCount,warningCount,defaultCount,debugCount,observationCount,inputTokens,outputTokens,totalTokens
0,77b0177257c16cbe9aade93a41bf308f,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:18:40.002Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.101804999999,DEFAULT,0,0,1,0,1,31170,553,31723
1,d0ed5e6524e3c8d7b08b93b4b56683fc,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:18:13.990Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.039588,DEFAULT,0,0,1,0,1,12671,105,12776
2,ef18ba0eb8cd33ed1eca332a002b2709,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:17:45.110Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.039288,DEFAULT,0,0,1,0,1,12296,160,12456
3,6af8cb2e4de0dae7701cf8f65bd6398f,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:16:34.734Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.063591,DEFAULT,0,0,1,0,1,19332,373,19705
4,2c0faab6db8de19a4d13f7e9dcd98dc7,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:15:58.473Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.059271,DEFAULT,0,0,1,0,1,18467,258,18725
5,8ad986dc707c8a546f2f89e530636adf,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:15:31.199Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.03372,DEFAULT,0,0,1,0,1,10610,126,10736
6,8fc335aa5b97039a0cf1dd32c78b6c79,cmgt9niok000wvo07s9ywpqc3,2026-02-17T05:12:41.313Z,[],False,Invoke Agent,9.0 Hotfix,None,user@email.com,staging,...,0.033141,DEFAULT,0,0,1,0,1,10102,189,10291


In [7]:
import json
import pandas as pd

# Load JSON
with open("test_dataset/case_create.json", "r") as f:
    data = json.load(f)

records = []

for item in data:
    uuid = item.get("uuid")
    
    # overall timestamps
    start_time = item.get("start_time")
    end_time = item.get("end_time")
    
    # compute total latency if possible
    total_time = None
    if start_time and end_time:
        total_time = pd.to_datetime(end_time) - pd.to_datetime(start_time)
        total_time = total_time.total_seconds()
    
    # loop through trace steps
    for step in item.get("trace", []):
        step_name = step.get("step")
        step_start = step.get("start")
        step_end = step.get("end")
        
        step_time = None
        if step_start and step_end:
            step_time = pd.to_datetime(step_end) - pd.to_datetime(step_start)
            step_time = step_time.total_seconds()
        
        records.append({
            "uuid": uuid,
            "step": step_name,
            "step_start": step_start,
            "step_end": step_end,
            "step_time_sec": step_time,
            "total_time_sec": total_time
        })

# Create DataFrame
df = pd.DataFrame(records)

df

""
